# Lesson 14 Lab — bitsandbytes 4-Bit Loading: NF4, Compute Dtype, and Nested Quantization

**Puzzle:** Does `load_in_4bit=True` specify how the layer computes?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A bitsandbytes 4-bit configuration contains at least storage codebook (`NF4` or FP4), compute dtype, optional double/nested quantization, and the module/backend that consumes it.

### Core mechanism

NF4 assigns its 16 codes non-uniformly rather than at equal integer spacing. During a linear operation the packed codes are dequantized or consumed by a fused path while activations use the configured compute dtype.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "14-bitsandbytes-4bit"
device = require_cuda()
torch.manual_seed(2026 + 14)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Nested quantization reduces scale metadata but does not halve activation precision. NF4 can suit normally distributed training weights, while inference latency depends on the installed kernel and shapes.

### What this code tests

The lab isolates codebook reconstruction and separately records package presence, so a numerical NF4 result cannot masquerade as bitsandbytes execution.

**Experiment:** Compare a reference NF4 codebook with uniform INT4 on normally distributed weights and probe whether bitsandbytes is installed.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import importlib.util
nf4=torch.tensor([-1.0,-0.6962,-0.5251,-0.3949,-0.2844,-0.1848,-0.0911,0.0,0.0796,0.1609,0.2461,0.3379,0.4407,0.5626,0.7230,1.0],device=device)
w=torch.randn(1_000_000,device=device); scale=w.abs().max(); normalized=(w/scale).clamp(-1,1)
idx=(normalized[:,None]-nf4[None,:]).abs().argmin(1); nf4_dq=nf4[idx]*scale
_,_,uniform=symmetric_quantize(w.reshape(1000,1000),bits=4,group_size=1000); uniform=uniform.reshape(-1)
result=base_result(14,"numerical-model"); result.update({"bitsandbytes_installed":importlib.util.find_spec("bitsandbytes") is not None,
    "nf4_error":error_metrics(w,nf4_dq),"uniform_int4_error":error_metrics(w,uniform),
    "conclusion":"Codebook behavior was measured numerically; bitsandbytes native execution is claimed only when installed."})


## 3. Inspect the evidence

The numerical comparison explains codebooks. Only an installed bitsandbytes layer would support a native-backend claim.

### Acceptance and rollback gate

Capture `BitsAndBytesConfig`, package/CUDA compatibility, actual module class, storage bytes, operator evidence, output regression, and timing.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "bitsandbytes_installed": false,
  "conclusion": "Codebook behavior was measured numerically; bitsandbytes native execution is claimed only when installed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:42+00:00",
  "lesson": 14,
  "nf4_error": {
    "cosine": 0.99187696,
    "mae": 0.10956612,
    "max_abs": 0.71935964,
    "rmse": 0.12783626
  },
  "schema_version": 1,
  "uniform_int4_error": {
    "cosine": 0.99001992,
    "mae": 0.12267612,
    "max_abs": 0.33905974,
    "rmse": 0.14239575
  }
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Record quantization type, compute dtype, nested-quant setting, and actual module class together.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).